# N1-Fast · ONE unchanged582888-param neural architecture · fresh30 · A100

P1–P4 missed RMSE<0.98/iRMSE<3.2. Keep N1 Fusion512, Jet-context128, U-PIR,
S4=4NFE/fine0. ONLY deployment operator rewrites + TensorRT builder level5.
No claim of a new neural architecture, quality gain or guaranteed3.9ms.
Preflight existing N1 checkpoint FIRST: all400 + original-vs-fast same weights.
If speed/quality misses, STOP before spending another30 epochs. Epoch resume is supported.
Training=PyTorch BF16/AdamW fused, metric KD ON/relative OFF, seed42/B4/1600+400.
Primary runtime=synchronized wall including D2D,100warmup/300iterations/100images.
Strict RMSE<0.98/iRMSE<3.2/mean≤3.9ms; reverse-order repeat must also meet speed cap.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib,json,shutil,subprocess,sys,uuid
from IPython.display import display
DRIVE=Path('/content/drive/MyDrive');DATA_ROOT=DRIVE/'GeoLift_Data'
NAME='AnchorFlow_S4_UPIR_N1Fast_30ep_A100'
REFERENCE_RUN=DRIVE/'GeoLift_RT_Runs/S4_UPIR_M3_F512_Fresh30_pair01'
RUN_TAG='S4_UPIR_N1Fast_Fresh30_pair01'  # SAME for resume
RUN=DRIVE/'GeoLift_RT_Runs'/RUN_TAG
assert RUN.resolve()!=REFERENCE_RUN.resolve()
frozen=RUN/'frozen_bundle';BUNDLE_OVERRIDE=''
if (frozen/'bundle_manifest.json').is_file():
    BUNDLE=frozen;print('RESUME: original frozen source selected')
else:
    assert not list(RUN.glob('*/last.pth')),'Restore original frozen_bundle before resume'
    choices=[Path(BUNDLE_OVERRIDE)] if BUNDLE_OVERRIDE else [DRIVE/NAME,DATA_ROOT/NAME]
    hits=[p for p in choices if (p/'bundle_manifest.json').is_file()]
    assert len(hits)==1,f'Upload ONE whole bundle: checked {choices}; found {hits}'
    BUNDLE=hits[0]
CODE=Path('/content/anchorflow_s4_n1fast_code');VARIANT='Q1_s4_n1_fast'
RUN.mkdir(parents=True,exist_ok=True);CODE.mkdir(parents=True,exist_ok=True)
CONFIG=RUN/'runtime_config.json'
assert (REFERENCE_RUN/'N1_s4_m3_f512/best_policy.pth').is_file(),'Set REFERENCE_RUN to old N1 run'
print('Source:',BUNDLE,'Outputs:',RUN)


In [ ]:
manifest = json.loads((BUNDLE/'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    if name.endswith('.ipynb'): continue  # Colab edits metadata/outputs
    source = BUNDLE/name
    assert source.is_file() and hashlib.sha256(source.read_bytes()).hexdigest()==expected, name
for name in [*manifest['files'], 'bundle_manifest.json']:
    source, target = BUNDLE/name, CODE/name
    target.parent.mkdir(parents=True, exist_ok=True)
    if name.endswith('.ipynb'):
        # Syntax contract reads the notebook locally. Metadata/output changes are
        # allowed; notebooks are excluded from the Python checkpoint source hash.
        shutil.copy2(source,target)
        continue
    if target.exists():
        assert target.read_bytes()==source.read_bytes(), f'Local source changed: {name}; use clean CODE/new run'
    else: shutil.copy2(source, target)
expected_py = {name for name in manifest['files'] if name.endswith('.py')}
actual_py = {p.relative_to(CODE).as_posix() for p in CODE.rglob('*.py') if '__pycache__' not in p.parts}
assert actual_py==expected_py, f'Extra/missing Python files: {actual_py^expected_py}'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-trt.txt')],check=True)
import torch, tensorrt
assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0), 'Select A100 runtime'
assert torch.cuda.is_bf16_supported(including_emulation=False), 'Native BF16 required'
assert tensorrt.__version__.startswith('10.13.3'), tensorrt.__version__
print(torch.__version__, torch.cuda.get_device_name(0), 'TensorRT', tensorrt.__version__)
test = subprocess.run([sys.executable,'-u','-m','unittest','discover','-s',str(CODE),
    '-p','test_contracts.py','-v'],cwd='/content',stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(test.stdout)
(RUN/'contracts.log').write_text(test.stdout)
assert test.returncode==0, 'Contracts failed; inspect full log above, do not skip tests'
display(json.loads((CODE/'model_profiles.json').read_text()))


## Data + original N1 provenance + real backward smoke

Keep old N1 frozen source/checkpoint/report/status on Drive; never edit them.
Fresh30 is NOT warm-started from trained N1. New sources/config live in a new run.


In [ ]:
cfg=json.loads((CODE/'config.json').read_text())
cfg.update(run_name=RUN_TAG,drive_data=str(DATA_ROOT),reference_run=str(REFERENCE_RUN))
required=('teacher_subset_2000/selected_2000_ids.json',
          'teacher_subset_2000/kitti_trainval_2000.tar',
          'teacher_subset_2000/metric_coarse_train_2000.tar',
          'test_1000/kitti_test_1000.tar')
missing=[str(DATA_ROOT/n) for n in required if not (DATA_ROOT/n).is_file()]
assert not missing,'Missing data payloads (keep TAR intact):\n'+'\n'.join(missing)
for candidate in (Path('/content/anchorflow_s4_m3_f512_work'),Path('/content/anchorflow_s4_phasefusion_work'),Path('/content/anchorflow_s4_edge600_work'),Path('/content/anchorflow_s4_d4split_work'),
                  Path('/content/anchorflow_s4_upir_norel_work')):
    if all((candidate/n).is_file() for n in ('data_contract.json','bundle_ready.json','metric_train/teacher_report.json','test_ready.json')):
        cfg['work']=str(candidate);break
assert cfg['relative_enabled'] is False and cfg['relative_weight']==0
if CONFIG.is_file() and list(RUN.glob('*/last.pth')):
    old=json.loads(CONFIG.read_text())
    runtime={'drive_data','drive_runs','work','workers','prefetch_factor'}
    assert {k:v for k,v in old.items() if k not in runtime}=={k:v for k,v in cfg.items() if k not in runtime},'Resume recipe changed; restore original runtime config'
WORK=Path(cfg['work']);CONFIG.write_text(json.dumps(cfg,indent=2))

def command(action,variant=None):
    argv=[sys.executable,'-u',str(CODE/'run.py'),action,'--config',str(CONFIG)]
    if variant:argv+=['--variant',variant]
    logfile=RUN/f'{action}_{variant or "run"}.log'
    with logfile.open('a') as handle:
        proc=subprocess.Popen(argv,cwd=str(CODE),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:print(line,end='');handle.write(line);handle.flush()
        rc=proc.wait()
    if rc:raise RuntimeError(f'{action} failed; inspect {logfile}. No fallback or checkpoint/hash edits.')

ready=all((WORK/n).is_file() for n in ('bundle_ready.json','metric_train/teacher_report.json','test_ready.json'))
assert shutil.disk_usage('/content').free>(4 if ready else 30)*2**30,'Insufficient SSD'
command('prepare')
command('history')  # read-only old N1 provenance; no old optimizer
command('smoke',VARIANT)
smoke=json.loads((RUN/VARIANT/'smoke_report.json').read_text())
proof=json.loads((RUN/'history_provenance.json').read_text())
assert smoke['passed'] and smoke['encoder_pretrained'] and smoke['coarse']['nfe']==4 and smoke['fine']['nfe']==0
assert smoke['shared_initial_sha256']==proof['initial_shared_sha256'],'Actual shared ImageNet initialization differs'
snapshot=RUN/'frozen_bundle'
for name in [*manifest['files'],'bundle_manifest.json']:
    target=snapshot/name;target.parent.mkdir(parents=True,exist_ok=True)
    if name.endswith('.ipynb'):
        if not target.exists():shutil.copy2(BUNDLE/name,target)
        continue
    if target.exists():assert target.read_bytes()==(BUNDLE/name).read_bytes(),f'Frozen snapshot changed: {name}'
    else:shutil.copy2(BUNDLE/name,target)
shutil.copy2(WORK/'data_contract.json',RUN/'data_contract.json')
display(smoke)


## Preflight existing trained N1 BEFORE long jobs

Two engines, same checkpoint/inputs: ORIGINAL FP32 vs FAST FP32. All400 accuracy,
fixed mean1mm/max6cm gates vs ORIGINAL PyTorch FP32, native-BF16 metric drift gates.
Level3/5 are explicit build-policy changes, not a fair isolated claim about one rewrite.
No FP16/TF32, resolution change, hidden reduction or fastest-repeat selection.


In [ ]:
(RUN/'benchmark_session.json').write_text(json.dumps({'session_id':uuid.uuid4().hex}))
command('preflight')
preflight=json.loads((RUN/'preflight_N1/trt_benchmark.json').read_text())
display(preflight['targets'])
if not (RUN/VARIANT/'last.pth').is_file():
    assert preflight['targets']['all_passed'] and preflight['targets']['stable_speed_target_passed'], (
        'Preflight missed target: NO fresh30 launched. Inspect full report; do not loosen parity gates. '
        'A new training run cannot fix a deployment speed failure.')
else:
    print('Existing epoch checkpoint: resume permitted; final benchmark must requalify all targets.')


## Optional requested fresh30 / exact epoch-boundary resume

Existing N1 weights already work with fastpath; retraining is NOT mathematically
required for the operator upgrade. This cell runs the requested matched fresh30
confirmation, never old N1/M3/P1–P4. No early stop; completed runs do no more steps.


In [ ]:
command('train',VARIANT)
display(json.loads((RUN/VARIANT/'training_status.json').read_text()))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(15,4))
for name,path in [('N1 previous',REFERENCE_RUN/'N1_s4_m3_f512/train_log.csv'),
                  ('Q1 fresh30',RUN/VARIANT/'train_log.csv')]:
    frame=pd.read_csv(path)
    assert len(frame)==30 and (frame['loss_weighted_relative']==0).all()
    for ax,key in zip(axes,('val_rmse_m','val_irmse_km_inv','val_boundary3_rmse_m')):
        ax.plot(frame['epoch']+1,frame[key],label=name);ax.set_title(key);ax.grid(alpha=.2)
axes[0].legend();plt.tight_layout();plt.show()


## FINAL: remeasure old N1 AND fresh30 in one current session

Each checkpoint: original+fast engines, all400 GT, same timing protocol and reversed
repeat. Old timings remain historical only. New fast model exports1000 test PNGs
and10 RGB/depth pairs. No public test GT, no independent full-KITTI evidence.


In [ ]:
(RUN/'benchmark_session.json').write_text(json.dumps({'session_id':uuid.uuid4().hex}))
command('preflight')  # REBENCH OLD N1 now, no optimizer/checkpoint edits
command('benchmark',VARIANT)
command('compare')
display(pd.read_csv(RUN/'comparison.csv'))
display(json.loads((RUN/'baseline_candidate.json').read_text()))
print('1000 predictions +10pairs:',RUN/VARIANT)
print('Frozen resume source:',RUN/'frozen_bundle')
